In [1]:
import cv2
import torch
import ultralytics
import torchvision
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
from ultralytics import solutions
from ultralytics import YOLO

In [3]:
import os

In [4]:
import shutil

In [5]:
import torch
import torch.nn as nn

In [6]:
# импорт необходимых библиотек

In [6]:
class ThickDigitCNN(nn.Module):
    def __init__(self):
        super(ThickDigitCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=5, padding=2) 
        self.conv2 = nn.Conv2d(32, 64, kernel_size=5, padding=2)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.dropout1 = nn.Dropout(0.25)
        self.dropout2 = nn.Dropout(0.5)
        self.fc1 = nn.Linear(128 * 16 * 16, 512) 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout2(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [7]:
# модель сверточной нейронной сети для распознавания цифр

In [8]:
def mask_color_create(frame):
    """
    Обрезаем подаваемый фрейм cv2.imread(путь к изображению)
    и определяем число пикселей нужного цвета в маске
    если число пикселей нужного цвета находится в пределах определенного значения,
    выводим True и после этого продолжаем обработку изображения, инача False - самокат не наш
    """
    # cropp_frame = cropper(frame)

    # словарь: ключ - название цветового канала, значение - число пикселей данного канала на изображении
    colors_pixels = dict()

    # переводим в удобоваримый для cv2 формат - HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Устанавливаем крайиние границы для фиолетового и синего цвета
    lower_purple = np.array([115, 100, 100])
    upper_purple = np.array([150, 255, 255])
    
    lower_blue = np.array([104, 117, 92]) 
    upper_blue = np.array([114, 255, 255])

    # Применяем маски для изображения переконвертированного в hsv 
    mask_purple = cv2.inRange(hsv, lower_purple, upper_purple)
    mask_blue = cv2.inRange(hsv, lower_blue, upper_blue)

    # cv2.imshow('Blue mask', mask_blue)
    # resp = cv2.bitwise_and(frame, frame, mask=mask_blue)
    # cv2.imshow('result', resp)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # Считаем число ненулевых пикселей по каждой маске
    purp_pixels = cv2.countNonZero(mask_purple)
    bl_pixels = cv2.countNonZero(mask_blue)

    # Заполням словарь
    colors_pixels['purple'] = purp_pixels
    colors_pixels['blue'] = bl_pixels

    # True, если число пикселей больше 1000, инчае - False
    flag_blue = True if colors_pixels['blue'] > 2000 else False
    flag_purple = True if colors_pixels['purple'] > 2000 else False

    return flag_blue | flag_purple

In [9]:
def activate(path_to_cropp: str, path_cropped: str = 'cropped-detections/'):
    """
    Функция, которая получит изображение, применит модель YOLO для детекции
    Добавит его в директорию с обрезанными изображениями
    Вызовет функцию для классификации, выведет ее значение
    После - удалит файл 
    """
    # Обертка, которая применит модель детекции и возьмет отдельные рамки для изображений
    cropper = solutions.ObjectCropper(
    model='runs/detect/train/weights/last_detectio.pt',
    )
    to_cropp = cropper(cv2.imread(path_to_cropp))
    total = dict()

    cropped_items = []
    # Цикл, в котором обрабатываются изображения из директории path_cropped, куда сохраняются обрезанные изображения
    for i in range(len(os.listdir(path_cropped))):
        cropped_item = os.path.join(path_cropped, os.listdir(path_cropped)[i])
        cropped_items.append(cropped_item)
        frame = cv2.imread(cropped_item)
        data = mask_color_create(frame=frame)
        total[i] = data
    # Удаляем директорию
    # shutil.rmtree(path_cropped)
    
    return total

In [10]:
# Модель сегментьации номера

In [11]:
model_seg = YOLO('segment_upd/best_segment.pt')

In [12]:
from imutils import contours

In [13]:
import torchvision.transforms.v2 as transforms

In [15]:
path_to_image_no = 'code_detevtion-4/test/images/IMG_4885_jpg.rf.db98a7a6c8a3aaa6a7d1cc832a0a4af1.jpg'

In [16]:
activate(path_to_image_no)

Ultralytics Solutions: ✅ {'region': None, 'show_in': True, 'show_out': True, 'colormap': None, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'json_file': None, 'records': 5, 'model': 'runs/detect/train/weights/last_detectio.pt'}


QApplication: invalid style override 'kvantum' passed, ignoring it.
	Available styles: Windows, Fusion



0: 640x480 2 samokats, 102.1ms
Speed: 4.3ms preprocess, 102.1ms inference, 9.9ms postprocess per image at shape (1, 3, 640, 480)
🚀 Results: SolutionResults(total_crop_objects=2)


{0: True, 1: True}

In [17]:
#Выдало True - самокат наш

In [18]:
model = ThickDigitCNN()  # Инициализация архитектуры

# 2. Загружаем веса (правильный способ)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
state_dict = torch.load('segment_upd/best_model_thick_digits_3.pth', map_location=device)
model.load_state_dict(state_dict)  # Загружаем параметры в модель

# 3. Переводим модель в режим оценки
model.eval()  # Теперь можно вызывать eval(), так как это объект модели
model.to(device) 

ThickDigitCNN(
  (conv1): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout1): Dropout(p=0.25, inplace=False)
  (dropout2): Dropout(p=0.5, inplace=False)
  (fc1): Linear(in_features=32768, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
)

In [19]:
# Работа с сегментированным объектом

In [19]:
results = model_seg.predict(source='cropped-detections/crop_1.jpg', save_crop=False)


image 1/1 /home/vad/pythonDir/jupyter/samokat/cropped-detections/crop_1.jpg: 640x480 2 codes, 51.5ms
Speed: 1.4ms preprocess, 51.5ms inference, 1.4ms postprocess per image at shape (1, 3, 640, 480)


In [25]:
from pathlib import Path
import numpy as np
# Iterate detection results 
for r in results:
    img = np.copy(r.orig_img)
    img_name = Path(r.path).stem

    # Iterate each object contour 
    for ci, c in enumerate(r):
        label = c.names[c.boxes.cls.tolist().pop()]

        b_mask = np.zeros(img.shape[:2], np.uint8)

        # Create contour mask 
        contour = c.masks.xy.pop().astype(np.int32).reshape(-1, 1, 2)
        _ = cv2.drawContours(b_mask, [contour], -1, (255, 255, 255), cv2.FILLED)

        # Choose one:

        # OPTION-1: Isolate object with black background
        mask3ch = cv2.cvtColor(b_mask, cv2.COLOR_GRAY2BGR)
        isolated = cv2.bitwise_and(mask3ch, img)

        # OPTION-2: Isolate object with transparent background (when saved as PNG)
        isolated = np.dstack([img, b_mask])

        # OPTIONAL: detection crop (from either OPT1 or OPT2)
        x1, y1, x2, y2 = c.boxes.xyxy.cpu().numpy().squeeze().astype(np.int32)
        iso_crop = isolated[y1:y2, x1:x2]

        _ = cv2.imwrite(f"{img_name}_{label}-{ci}.png", iso_crop)

In [41]:
image = cv2.imread('crop_1_code-1.png')
image = cv2.resize(image, (1050, 1610))
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)
cnts = cv2.findContours(binary, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
cnts, _ = contours.sort_contours(cnts[0])

dig_ims = []
for c in cnts:
    area = cv2.contourArea(c)
    x, y, w, h = cv2.boundingRect(c)
    if 5_000 < area < 21_000:
        print(area)
        dig_ims.append((x, y, w, h))

dig_ims.sort(key=lambda c: (c[1], c[0]))
i = 0
for (x, y, w, h) in dig_ims:
    i += 1
    margin = 20
    img = image[max(0,y-margin):min(image.shape[0],y+h+margin),
               max(0,x-margin):min(image.shape[1],x+w+margin)]

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    _, binary = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)
    
    kernel = np.ones((2,2), np.uint8)
    
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    thin = cv2.erode(closed, kernel, iterations=5)

    cv2.imshow(f'contours_{i}.png', thin)
    cv2.imwrite(f'cropped_number{i}.png', thin)

    cv2.waitKey(0)
    cv2.destroyAllWindows()

# for (x, y, w, h) in dig_ims:
#     i += 1
#     margin = 5
#     img = image[max(0,y-margin):min(image.shape[0],y+h+margin),
#                max(0,x-margin):min(image.shape[1],x+w+margin)]

#     # Конвертация и размытие
#     gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
#     blurred = cv2.GaussianBlur(gray, (5,5), 0)
    
#     # Адаптивная бинаризация
#     binary = cv2.adaptiveThreshold(blurred, 255, 
#                                   cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
#                                   cv2.THRESH_BINARY_INV, 11, 2)
    
#     # Морфологическое закрытие с большим ядром
#     kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
#     closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=2)
    
#     # Медианный фильтр
#     smooth = cv2.medianBlur(closed, 3)
    
#     # Аппроксимация контуров
#     contours, _ = cv2.findContours(smooth, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#     approx_contours = []
#     for cnt in contours:
#         epsilon = 0.005 * cv2.arcLength(cnt, True)
#         approx = cv2.approxPolyDP(cnt, epsilon, True)
#         approx_contours.append(approx)
    
#     # Создаем гладкое изображение
#     smooth_contour = np.zeros_like(smooth)
#     cv2.drawContours(smooth_contour, approx_contours, -1, 255, thickness=cv2.FILLED)
    
#     # Сохраняем результат
#     cv2.imwrite(f'cropped_smooth_{i}.png', smooth_contour)
#     cv2.imshow(f'Smooth {i}', smooth_contour)
#     cv2.waitKey(0)
#     cv2.destroyAllWindows()

7694.5
10434.0
14390.0
15023.0


In [42]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import cv2
import numpy as np
from PIL import Image, ImageFilter, ImageOps
from imutils import contours

In [43]:
# необходимые трансоформации перед подачей в модель

In [50]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.Lambda(lambda x: x.filter(ImageFilter.MaxFilter(9))),  # Утолщение
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

def predict_digit(image_path):
    # Открываем и преобразуем изображение
    img = Image.open(image_path).convert('L')  # Конвертируем в ч/б
    
    # Применяем трансформации
    img_transformed = transform(img)
    img_transformed = img_transformed.unsqueeze(0)  # Добавляем batch-размерность
    model.zero_grad()
    # Подаем в модель
    with torch.no_grad():
        output = model(img_transformed)
        prediction = torch.argmax(output, dim=1).item()
    
    return prediction

In [51]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.v2 as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import cv2
import numpy as np
from PIL import Image, ImageFilter, ImageOps
from imutils import contours

In [52]:
code_number = ''
for i in range(1, 5):
    image_path = f'cropped_number{i}.png'
    predicted_digit = predict_digit(image_path)
    code_number += str(predicted_digit)

In [53]:
int(code_number)

8808

In [33]:
# Всё сходится. Пайплайн отработал верно!


In [27]:
import pytesseract

In [69]:
image = cv2.resize(cv2.imread('crop_1_code-0.png'), (140, 100))
gray =cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
binary = cv2.threshold(gray, 0,255,cv2.THRESH_BINARY| cv2.THRESH_OTSU)[1]

In [70]:
cv2.imwrite('1test_gray_crop.png', binary)

True

In [71]:
text = pytesseract.image_to_string(Image.open('1test_gray_crop.png'))

In [72]:
text

'rated\n\n..\n\\ 2 a!\n'

In [96]:
image = cv2.resize(cv2.imread('cropped_number2.png'), (140, 100))
gray =cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
binary = cv2.threshold(gray, 0,255,cv2.THRESH_BINARY| cv2.THRESH_OTSU)[1]
cv2.imwrite('gray_crop2.png', binary)

True

In [75]:
text = pytesseract.image_to_string(Image.open('gray_crop2.png'), config='--psm 5')

In [76]:
text

'&\n'

In [104]:
# import the necessary packages
import numpy as np
import cv2
import imutils

In [109]:
image = cv2.imread("cropped_number2.png")
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (7, 7), 0)

# threshold the image
ret,thresh1 = cv2.threshold(gray ,127,255,cv2.THRESH_BINARY_INV)

# dilate the white portions
dilate = cv2.dilate(thresh1, None, iterations=2)

# find contours in the image
cnts = cv2.findContours(dilate.copy(), cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE)
cnts = cnts[0] if imutils.is_cv2() else cnts[1]

orig = image.copy()
i = 0

for cnt in cnts:
    # Check the area of contour, if it is very small ignore it
    if(cv2.contourArea(cnt) < 10):
        continue

    # Filtered countours are detected
    x,y,w,h = cv2.boundingRect(cnt)

    # Taking ROI of the cotour
    roi = image[y:y+h, x:x+w]

    # Mark them on the image if you want
    cv2.rectangle(orig,(x,y),(x+w,y+h),(0,255,0),2)

    # Save your contours or characters
    cv2.imwrite("roi" + str(i) + ".png", roi)

    i = i + 1 

cv2.imshow("Image", orig) 
cv2.waitKey(0)

error: OpenCV(4.11.0) /io/opencv/modules/imgproc/src/shapedescr.cpp:315: error: (-215:Assertion failed) npoints >= 0 && (depth == CV_32F || depth == CV_32S) in function 'contourArea'


In [112]:
image = cv2.resize(cv2.imread("cropped_number2.png"), (140, 100))
if image is None:
    raise ValueError("Не удалось загрузить изображение. Проверьте путь к файлу.")

# Преобразование в grayscale и размытие
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (7, 7), 0)

# Бинаризация
ret, thresh1 = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)

# Расширение белых областей
dilate = cv2.dilate(thresh1, None, iterations=2)

# Поиск контуров
cnts = cv2.findContours(dilate.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts = imutils.grab_contours(cnts)  # Универсальный способ получения контуров

orig = image.copy()
i = 0

for cnt in cnts:
    # Проверка, что контур содержит достаточно точек
    if len(cnt) < 2:  # Минимум 5 точек для корректного расчета площади
        continue
        
    try:
        area = cv2.contourArea(cnt)
        if area < 10:
            continue
            
        # Получаем bounding box
        x, y, w, h = cv2.boundingRect(cnt)
        
        # Получаем ROI
        roi = image[y:y+h, x:x+w]
        
        # Рисуем прямоугольник
        cv2.rectangle(orig, (x,y), (x+w,y+h), (0,255,0), 2)
        
        # Сохраняем ROI
        cv2.imwrite(f"roi_{i}.png", roi)
        i += 1
        
    except Exception as e:
        print(f"Ошибка обработки контура: {e}")
        continue

cv2.imshow("Result", orig)
cv2.waitKey(0)
cv2.destroyAllWindows()